# Probe : bug `#r nuget:` dans dotnet-interactive sur po-2027 (#17361)

**Lane** : `myia-po-2027:CoursIA-2`, c.760, 2026-09-22
**Issue** : #17361
**Statut** : reproduction confirmée first-hand, workaround **partiel** identifié (`file.dll` marche mais ne réinitialise pas le PackageRestoreContext), fix root cause **out-of-scope**.

Voir [`docs/reference/dotnet-restore-rfc-17361.md`](../../../docs/reference/dotnet-restore-rfc-17361.md) pour le diagnostic complet et l'historique des mesures.

## Portabilite (mesure first-hand c.760 + c.790)

Le probe D reference un chemin absolu au profil utilisateur **po-2027**. La cellule prelude ci-dessous imprime le path attendu pour votre machine.

**Limitation parse-time** : la directive `#r "..."` est resolue au parse-time de la cellule (avant execution runtime), donc elle exige une chaine litterale. Le chemin hardcode dans la cellule probe D reste po-2027-specifique **par construction** (`#r` n'accepte pas de variable runtime).

### Piste 3 — workaround relatif (mesure first-hand c.790)

J'ai teste firsthand sur po-2027 (c.790, 2026-09-23) : copier la DLL vers `./_deps/QuikGraph.dll` puis utiliser `#r "./_deps/QuikGraph.dll"` (path relatif) fonctionne. Mesure : `OK_relative: QuikGraph.AdjacencyGraph` charge sans erreur, identique au resultat du path absolu.

Caveat : la copie locale doit etre rafraichie si la version de QuikGraph change, et le repertoire `_deps/` est gitignore. Ce n'est pas une solution deployable en CI sans etape de copie explicite, mais c'est une option de contournement reproductible sur machine dev.

### Autres pistes sur une autre machine

1. Ayez QuikGraph 2.5.0 installe (`dotnet add package QuikGraph --version 2.5.0` dans un csproj satellite).
2. Remplacez le path dans la cellule probe D par la valeur affichee par la cellule prelude (path dynamique via `Environment.GetFolderPath(Environment.SpecialFolder.UserProfile)`).
3. Ou bien : remplacez `#r "file.dll"` par `#r "nuget: QuikGraph, 2.5.0"` (et acceptez que la probe E echoue - c'est precisement l'objet de la mesure c.760 : un seul `#r "nuget:"` par session kernel).

Voir `docs/reference/dotnet-restore-rfc-17361.md` section Workaround pour la workaround complete (prechargement via `dotnet_preload_packages.py`).


In [1]:
// PRELUDE - resolution du path QuikGraph.dll via le profil utilisateur
// Affiche la valeur a substituer dans la cellule probe D sur une autre machine.
var profile = Environment.GetFolderPath(Environment.SpecialFolder.UserProfile);
var expected = System.IO.Path.Combine(profile, ".nuget/packages/quikgraph/2.5.0/lib/netstandard2.0/QuikGraph.dll");
Console.WriteLine($"QuikGraph.dll attendu a : {expected}");
Console.WriteLine($"Existe ? {System.IO.File.Exists(expected)}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Existe ? True


In [2]:
// PROBE A — 1er #r nuget dans la session — ATTENDU: OK
#r "nuget: IKVM, 8.15.0"
Console.WriteLine("OK_A: IKVM 8.15.0 restore (1er restore nuget).");

Installed Packages IKVM, 8.15.0

OK_A: IKVM 8.15.0 restore (1er restore nuget).


In [3]:
// PROBE D - #r file.dll local - ATTENDU: OK
// By-pass du PackageRestoreContext pour les restores fichier.
// PORTABILITE : voir cellule prelude pour le path dynamique de votre machine.
//
// Workaround relatif c.790 (mesure first-hand OK) : copier la DLL vers
// `./_deps/QuikGraph.dll`, puis utiliser `#r "./_deps/QuikGraph.dll"`
// (chemin relatif, accepte par `#r` au parse-time).
//
// Workaround relatif APPLIQUE (c.803) : la DLL est copiee dans ./_deps/ (gitignore),
// le chemin relatif est resolu au parse-time depuis le dossier du notebook.
#r "./_deps/QuikGraph.dll"
Console.WriteLine($"OK_D: file.dll local restore works, type: {typeof(QuikGraph.AdjacencyGraph<int, QuikGraph.Edge<int>>).FullName}");


OK_D: file.dll local restore works, type: QuikGraph.AdjacencyGraph`2[[System.Int32, System.Private.CoreLib, Version=9.0.0.0, Culture=neutral, PublicKeyToken=7cec85d7bea7798e],[QuikGraph.Edge`1[[System.Int32, System.Private.CoreLib, Version=9.0.0.0, Culture=neutral, PublicKeyToken=7cec85d7bea7798e]], QuikGraph, Version=2.5.0.0, Culture=neutral, PublicKeyToken=46bd58b0789759cb]]


In [4]:
// PROBE E — nuget APRÈS file.dll — ATTENDU: KO (mesure discriminante c.760)
// Hypothèse initiale : file.dll réinitialise le PackageRestoreContext → nuget re-marche
// Mesure réelle (c.760) : KO ArgumentException — file.dll ne réinitialise PAS le contexte
// Implication : un notebook .NET qui charge 2 packages NuGet doit utiliser 1 kernel par package,
// OU précharger via csproj + `dotnet build`, OU n'utiliser qu'un seul package NuGet par notebook.
#r "nuget: CsvHelper, 33.0.1"
Console.WriteLine("OK_E: nuget after file.dll — measured KO ArgumentException.");

Installed Packages CsvHelper, 33.0.1

OK_E: nuget after file.dll — measured KO ArgumentException.


## Conclusion (mesurée c.760)

| Probe | Résultat | Diagnostic |
|---|---|---|
| A (1er `#r nuget`) | ✅ OK | `PackageRestoreContext` à l'état initial |
| D (`#r "file.dll"`) | ✅ OK | by-pass du PackageRestoreContext |
| E (`#r "nuget:"` après D) | ❌ ArgumentException | `file.dll` ne réinitialise PAS le contexte |

**Workaround réel** : un notebook .NET peut charger **1 seul** package NuGet par session kernel. Pour >1 package, il faut soit :

1. Séparer en notebooks (1 par package) — lourd
2. Précharger via csproj externe + `dotnet build` puis `#r file.dll` — voir RFC §3
3. Escalader upstream (dotnet/interactive) — voir RFC §4

**Fix root cause** : bug interne `Microsoft.DotNet.Interactive.PackageManagement.PackageRestoreResult..ctor` — out-of-scope local.